In [ ]:
# !pip install "arcade==3.3.*"

In [ ]:
import arcade

print("arcade version:", getattr(arcade, "VERSION", None) or arcade.__version__)
print("If ^this is 3.3.x, yours is ok.")

In [ ]:
import os

os.makedirs("demos", exist_ok=True)
print("Demos will be saved in:", os.path.abspath("demos"))

## Section 1 — Windows, classes, and `self`

### Python first: what is a class?

Every arcade game is a **class**. 

A **class** is a blueprint. An **object** is one thing built from that blueprint.

- `dict` is a class. `{"a": 1}` is an object.
- `Robot` (below) is a class. `r1` and `r2` are two separate objects.

Three pieces of vocabulary:

- **`__init__`** — the *setup* method. Python calls it automatically when you build an object.
  It is where you give the object its starting data. (Those are two underscores on each side.)
- **`self`** — the object *currently being worked on*. Inside the class you always write `self.x`
  to mean "this particular object's `x`." It is the first parameter of every method, and you
  never pass it in yourself — Python does that for you.
- **method** — a function that lives inside a class.

Run this cell. No arcade involved:

In [ ]:
class Robot:
    def __init__(self, name, energy=100):
        # This runs once, when the object is created
        # 'self' is the new object. These lines give it its starting data
        self.name = name
        self.energy = energy

    def work(self, amount):
        # A method: a function that belongs to the object.
        self.energy = self.energy - amount
        return f"{self.name} worked. Energy now {self.energy}."

r1 = Robot("Bolt")
r2 = Robot("Rusty", energy=50)

print(r1.work(30))
print(r2.work(30))
print("r1 energy:", r1.energy, "| r2 energy:", r2.energy)

Notice the last line: `r1` and `r2` have **separate** energy. That is the point of `self`:
it keeps each object's data apart. Notice also that you called `r1.work(30)` with one argument,
but `work` is defined with two (`self, amount`). Python fills in `self` for you.

### Inheritance and `super()`

A class can be built **on top of** another class. It gets everything the parent has, and adds
or replaces pieces.

`super().__init__(...)` means: *"run the parent's setup first."* If you forget it, the parent
never gets set up, and you get errors that seem to come from nowhere.

In [ ]:
class MiningRobot(Robot):  # <- "MiningRobot is a Robot"
    def __init__(self, name, depth):
        super().__init__(name, energy=200)  # run Robot.__init__ first
        self.depth = depth  # then add something new

    def dig(self):
        self.energy -= 50
        return f"{self.name} dug to {self.depth}m. Energy {self.energy}"


m = MiningRobot("DigDugDoug", depth=12)
print(m.dig())
print(m.work(10))  # inherited from Robot for free


In [ ]:
%%writefile demos/01_window.py
import arcade

# Constants: settings you might want to change
SCREEN_WIDTH = 600
SCREEN_HEIGHT = 600
SCREEN_TITLE = "Arcade Window"

class MyGame(arcade.Window):
    def __init__(self):
        # Run arcade.Window's own setup first. This is what opens a window.
        super().__init__(SCREEN_WIDTH, SCREEN_HEIGHT, SCREEN_TITLE)

        # Now our own setup.
        self.background_color = arcade.color.DARK_BLUE_GRAY
    
    def on_draw(self):
        # Arcade calls this ~60 times a second
        self.clear()

        # A dot at the origin, to show where (0, 0) is
        arcade.draw_circle_filled(0, 0, 25, arcade.color.RED)
        arcade.draw_text("(0, 0) is here", 35, 20, arcade.color.WHITE, 14)

        # A dot at the top-right corner.
        arcade.draw_circle_filled(SCREEN_WIDTH, SCREEN_HEIGHT, 25, arcade.color.YELLOW)
        arcade.draw_text(
            f"({SCREEN_WIDTH}, {SCREEN_HEIGHT})",
            SCREEN_WIDTH - 130, SCREEN_HEIGHT - 45,
            arcade.color.WHITE, 14,
        )

# Outside the class. This creates one object and starts the looop.
window = MyGame()
arcade.run()

In [12]:
!python demos/01_window.py

/Users/rafalr/Desktop/~Repositories/TTA-Class-Files/2026 Term 4/L4212T3 - Sat/.venv/lib/python3.13/site-packages/pyglet/media/codecs/__init__.py:152: UserWarning: FFmpeg release version not found. This may be a new or untested release. Unknown behavior may occur. Versions Loaded: {'avcodec': 63, 'avformat': 63, 'avutil': 61, 'swresample': 7, 'swscale': 10}
  warnings.warn(f'FFmpeg release version not found. This may be a new or untested release. Unknown behavior may occur. Versions Loaded: {ffmpeg_lib.compat.versions}')
/Users/rafalr/Desktop/~Repositories/TTA-Class-Files/2026 Term 4/L4212T3 - Sat/.venv/lib/python3.13/site-packages/arcade/exceptions.py:138: PerformanceWarning: draw_text is an extremely slow function for displaying text. Consider using Text objects instead.
  warnings.warn(message, warning_type)


## Section 2 — Constants, numbers, and shapes

### Python first: constants and derived values

Imagine moving this house 50 pixels left:

```python
arcade.draw_rect_filled(arcade.XYWH(400, 220, 260, 200), BEIGE)   # body
arcade.draw_rect_filled(arcade.XYWH(400, 170, 60, 100), BROWN)    # door
arcade.draw_circle_filled(320, 260, 20, CYAN)                     # window
```

You must find and fix six numbers, and missing one breaks the house. These are **magic numbers**:
raw values whose relationships live only in your head. The fix:

1. **Name them.** Python has no true constants, so the convention is `UPPER_CASE` at the top of
   the file, meaning *"a setting — don't reassign it."*
2. **Derive the rest.** The door's x isn't `400`, it's *the same as the house's x*. Write that
   down as arithmetic and the computer maintains it for you.

Two small things you'll need: `/` gives a float (`5 / 2` → `2.5`) while `//` floors to an int
(`5 // 2` → `2`) — pixels are happy as floats. And a **tuple** is a fixed group in parentheses,
like the `(255, 0, 0)` of a color; you can unpack one with `r, g, b = my_color`.

In [ ]:
HOUSE_X = 400
HOUSE_Y = 220
HOUSE_W = 260
HOUSE_H = 200

door_x = HOUSE_X # door is centered on the house
door_w = HOUSE_W / 4
house_top = HOUSE_Y + HOUSE_H / 2
window_x = HOUSE_X - HOUSE_W / 3

print(f"house top edge: {house_top}")
print(f"door width:     {door_w}")
print(f"window x:       {window_x}")

# Tuples: a color is just three numbers grouped together
SAND = (194, 178, 128)
r, g, b = SAND
print(f"SAND -> red={r} green={g} blue={b}")

# / vs //
print("5 / 2 =", 5 / 2, " 5 // 2 =", 5 // 2)

house top edge: 320.0
door width:     65.0
window x:       313.3333333333333
SAND -> red=194 green=178 blue=128


In [1]:
%%writefile demos/02_shapes.py
import arcade

SCREEN_WIDTH = 800
SCREEN_HEIGHT = 600
TITLE = "Basic Shapes"

class BasicShapes(arcade.Window):
    def __init__(self):
        super().__init__(SCREEN_WIDTH, SCREEN_HEIGHT, TITLE)
        self.background_color = arcade.color.ASH_GREY

    def on_draw(self):
        self.clear()

        # --- Circles: x, y, radius ---
        arcade.draw_circle_filled(100, 480, 40, arcade.color.RED)
        arcade.draw_circle_outline(100, 480, 40, arcade.color.BLACK, border_width=3)

        # --- Rectangle: build a Rect, then draw it ---
        box = arcade.XYWH(260, 480, 90, 70)
        arcade.draw_rect_filled(box, arcade.color.BLUE)
        arcade.draw_rect_outline(box, arcade.color.BLACK, border_width=2)

        same_box = arcade.LBWH(440, 445, 90, 70) # left, bottom, width, height
        arcade.draw_rect_filled(same_box, arcade.color.DARK_GREEN)

        # --- Line: note line_width, not border_width ---
        arcade.draw_line(550, 445, 650, 515, arcade.color.DARK_GREEN, line_width=5)

        # --- Polygon: a list of (x, y) points ---
        diamond = [(720, 445), (760, 480), (720, 515), (680, 480)]
        arcade.draw_polygon_filled(diamond, arcade.color.YELLOW)
        arcade.draw_polygon_outline(diamond, arcade.color.BLACK, line_width=3)

        # --- Alpha: a translucent panel over everything above ---
        panel = arcade.LRBT(60, 780, 300, 400)
        arcade.draw_rect_filled(panel, (0, 0, 0, 120)) # 4th number = see-through
        arcade.draw_text(
            "translucent panel (alpha = 120)", 80, 340, arcade.color.WHITE, 18
        )
        arcade.draw_text("Basic Shapes", 60, 550, arcade.color.BLACK, 22)

window = BasicShapes()
arcade.run()

Overwriting demos/02_shapes.py


In [ ]:
!python demos/02_shapes.py

/Users/rafalr/Desktop/~Repositories/TTA-Class-Files/2026 Term 4/L4212T3 - Sat/.venv/lib/python3.13/site-packages/pyglet/media/codecs/__init__.py:152: UserWarning: FFmpeg release version not found. This may be a new or untested release. Unknown behavior may occur. Versions Loaded: {'avcodec': 63, 'avformat': 63, 'avutil': 61, 'swresample': 7, 'swscale': 10}
  warnings.warn(f'FFmpeg release version not found. This may be a new or untested release. Unknown behavior may occur. Versions Loaded: {ffmpeg_lib.compat.versions}')


In [ ]:
%%writefile demos/03_stamper.py
import arcade

SCREEN_WIDTH = 800
SCREEN_HEIGHT = 600

PALETTE = [
    arcade.color.RED,
    arcade.color.GOLD,
    arcade.color.CYAN,
    arcade.color.DARK_SPRING_GREEN,
]
RADIUS = 22

class Stamper(arcade.Window):
    def __init__(self):
        super().__init__(SCREEN_WIDTH, SCREEN_HEIGHT, "Shape Stamper")
        self.background_color = arcade.color.DARK_SLATE_GRAY

        # A plain Python list of (x, y, color) tuples
        self.stamps = []
        self.color_index = 0
        self.mouse_x = SCREEN_WIDTH / 2
        self.mouse_y = SCREEN_HEIGHT / 2

        self.hint = arcade.Text("click to stamp - SPACE changes color - C clears",
                                SCREEN_WIDTH / 2, 16, arcade.color.WHITE, 14,
                                anchor_x="center")
        self.count = arcade.Text("", 16, SCREEN_HEIGHT - 28, arcade.color.WHITE, 16)

        def on_mouse_motion(self, x, y, dx, dy):
            self.mouse_x = x
            self.mouse_y = y

        def on_mouse_press(self, x, y, button, modifiers):
            self.stamps.append((x, y, PALETTE[self.color_index]))

        def on_key_press(self, symbol, modifiers):
            if symbol == arcade.key.SPACE:
                # Wrap back to 0 after the last color - that is what % is for
                self.color_index = (self.color_index = 1) % len(PALETTE)
            elif symbol = arcade.key.C:
                self.stamps.clear()

        def on_draw(self):
            self.clear()
        